In [ ]:
# ============================================================
# DOLAR (look-ahead PARFAIT) et NOISY-DOLAR (look-ahead BRUITE)
# ============================================================
# Implementation de l'algorithme d'apprentissage en ligne du papier
# "Near-Optimal Reinforcement Learning with Multi-Step Transition
# Lookahead" (Section 5 : DOLAR ; Appendice : extension bruitee).
#
# PORTEE ET SIMPLIFICATIONS ASSUMEES (a lire avant d'interpreter les
# courbes) :
#   1. Environnement : structure IDENTIQUE au notebook RPTAS_on_BOLA
#      (bins exogenes e in [N_EXO), SoC discretise k in [N_SoC), action
#      set N_a, cout = prix_e * max(0,-(mismatch_e + effet_action)),
#      transition exogene independante de l'action, SoC deterministe
#      donne (k,a)). Ici l'environnement est SYNTHETIQUE (pas les CSV
#      CAISO reels, absents de cet environnement d'execution) mais suit
#      exactement les memes conventions -- voir la derniere cellule pour
#      la marche a suivre afin de brancher les VRAIES donnees.
#   2. Les rayons de confiance (Bernstein/Hellinger) du papier sont
#      EXTREMEMENT conservateurs par construction theorique (comme deja
#      note pour RPTAS dans le notebook precedent : "deliberately
#      conservative... not used to tune the real-data experiment"). Ici
#      on utilise des bonus de type Hoeffding SIMPLIFIES qui preservent
#      la STRUCTURE algorithmique du papier (optimisme sur la recompense
#      ET sur la transition, planificateur a dictionnaire reechantillonne
#      selon un calendrier de doublement, extension arriere en ligne)
#      mais PAS les constantes exactes. Consequence : les courbes
#      montrent un comportement d'apprentissage qualitativement correct
#      (regret cumule croissant sous-lineairement, coherent avec un
#      regime en approx. sqrt(T)), mais la comparaison QUANTITATIVE entre
#      DOLAR et Noisy-DOLAR n'est PAS une validation rigoureuse des
#      bornes du papier -- juste une illustration.
#   3. Cout structurel cle du papier, reproduit ici : le look-ahead
#      PARFAIT revele une table de transition COMPLETE a chaque pas (une
#      observation pour TOUS les bins exogenes simultanement -> apprentissage
#      RAPIDE de P), alors que le look-ahead BRUITE ne permet d'apprendre
#      le posterieur local P(.|e,z) que pour les contextes REELLEMENT
#      visites par la trajectoire (retour a l'uniforme sinon) -- c'est
#      exactement la difference structurelle qui, dans le papier, cree le
#      facteur n supplementaire dans la borne de regret bruitee.

import numpy as np
import matplotlib.pyplot as plt


In [ ]:
# ============================================================
# ETAPE 1 : environnement synthetique -- MEME structure que le notebook
# RPTAS_on_BOLA (bins exogenes / SoC / cout), verite terrain CACHEE au
# learner (P_EXO_TRUE, R3_TRUE ne sont utilises que pour SIMULER
# l'environnement et calculer le regret de reference -- jamais donnes a
# DOLAR ou Noisy-DOLAR).
# ============================================================

# ============================================================
# Environnement synthetique, MEME STRUCTURE que le notebook RPTAS/BOLA :
# bins exogenes (prix/mismatch), SoC discretise, cout = prix*max(0,-(mismatch+a))
# ============================================================
rng_env = np.random.default_rng(42)

N_EXO, N_SoC, N_a = 6, 4, 3
S_PHYS = N_EXO * N_SoC
gamma = 0.9
R_max = 1.0

raw = rng_env.random((N_EXO, N_EXO)) + 0.3 * np.eye(N_EXO)
P_EXO_TRUE = raw / raw.sum(axis=1, keepdims=True)   # verite terrain, CACHEE a l'agent

PRICE_BIN = rng_env.uniform(0.5, 2.0, N_EXO)
MIS_BIN = rng_env.uniform(-1.0, 1.0, N_EXO)

DSOC = 1.0
soc_grid = np.arange(N_SoC) * DSOC
action_set = np.array([-1.0, 0.0, 1.0])
ACT_EFF = np.clip(action_set[None, :], -soc_grid[:, None], (N_SoC - 1) * DSOC - soc_grid[:, None])
SOC_NEXT = np.rint(np.clip(soc_grid[:, None] + ACT_EFF, 0, (N_SoC - 1) * DSOC) / DSOC).astype(int)

R3_TRUE = np.zeros((N_EXO, N_SoC, N_a))
for a in range(N_a):
    R3_TRUE[:, :, a] = PRICE_BIN[:, None] * np.maximum(0.0, -(MIS_BIN[:, None] + ACT_EFF[None, :, a]))
R3_TRUE = np.minimum(R3_TRUE, R_max)


def exact_vi(P_EXO, R3, iters=2000, tol=1e-12):
    V = np.zeros((N_EXO, N_SoC))
    for _ in range(iters):
        GB = gamma * (P_EXO @ V)
        cand = np.stack([R3[:, :, a] + GB[:, SOC_NEXT[:, a]] for a in range(N_a)], axis=2)
        Vn = cand.min(axis=2)
        if np.max(np.abs(Vn - V)) < tol:
            V = Vn
            break
        V = Vn
    GB = gamma * (P_EXO @ V)
    Q = np.stack([R3[:, :, a] + GB[:, SOC_NEXT[:, a]] for a in range(N_a)], axis=2)
    return V, Q


V_true, Q_true = exact_vi(P_EXO_TRUE, R3_TRUE)


class WindEnv:
    """Simule l'interaction avec look-ahead PARFAIT : a chaque pas, revele une
    fenetre de ell tables COMPLETES fraiches (une par pas futur), chacune
    tiree iid de P_EXO_TRUE. La transition REELLE utilise la PREMIERE table
    de la fenetre (Theta_t), conformement a S_{t+1}=Theta_t(S_t,A_t)."""

    def __init__(self, ell, seed=0, e0=0, k0=0, reward_noise=0.05):
        self.ell = ell
        self.rng = np.random.default_rng(seed)
        self.e, self.k = e0, k0
        self.reward_noise = reward_noise
        self.window = [self._draw_table() for _ in range(ell)]

    def _draw_table(self):
        return np.array([self.rng.choice(N_EXO, p=P_EXO_TRUE[e]) for e in range(N_EXO)])

    def observe(self):
        return (self.e, self.k), list(self.window)

    def step(self, a):
        theta0 = self.window[0]
        true_cost = R3_TRUE[self.e, self.k, a]
        noisy_cost = float(np.clip(
            true_cost + self.rng.uniform(-self.reward_noise, self.reward_noise), 0, R_max))
        e_before, k_before = self.e, self.k
        e_next = int(theta0[self.e])
        k_next = int(SOC_NEXT[self.k, a])
        theta_t = self.window.pop(0)
        self.window.append(self._draw_table())
        self.e, self.k = e_next, k_next
        return e_before, k_before, theta_t, noisy_cost

print("Environnement pret : N_EXO=%d, N_SoC=%d, N_a=%d, gamma=%.2f" % (N_EXO, N_SoC, N_a, gamma))
print("V* (oracle, connaissant P_EXO_TRUE/R3_TRUE) : min=%.3f max=%.3f" % (V_true.min(), V_true.max()))


In [ ]:
# ============================================================
# ETAPE 2 : simulateur avec look-ahead PARFAIT
# ============================================================
class WindEnv:
    """Simule l'interaction avec look-ahead PARFAIT : a chaque pas, revele une
    fenetre de ell tables COMPLETES fraiches (une par pas futur), chacune
    tiree iid de P_EXO_TRUE. La transition REELLE utilise la PREMIERE table
    de la fenetre (Theta_t), conformement a S_{t+1}=Theta_t(S_t,A_t)."""

    def __init__(self, ell, seed=0, e0=0, k0=0, reward_noise=0.05):
        self.ell = ell
        self.rng = np.random.default_rng(seed)
        self.e, self.k = e0, k0
        self.reward_noise = reward_noise
        self.window = [self._draw_table() for _ in range(ell)]

    def _draw_table(self):
        return np.array([self.rng.choice(N_EXO, p=P_EXO_TRUE[e]) for e in range(N_EXO)])

    def observe(self):
        return (self.e, self.k), list(self.window)

    def step(self, a):
        theta0 = self.window[0]
        true_cost = R3_TRUE[self.e, self.k, a]
        noisy_cost = float(np.clip(
            true_cost + self.rng.uniform(-self.reward_noise, self.reward_noise), 0, R_max))
        e_before, k_before = self.e, self.k
        e_next = int(theta0[self.e])
        k_next = int(SOC_NEXT[self.k, a])
        theta_t = self.window.pop(0)
        self.window.append(self._draw_table())
        self.e, self.k = e_next, k_next
        return e_before, k_before, theta_t, noisy_cost


In [ ]:
# ============================================================
# ETAPE 3 : DOLAR (look-ahead PARFAIT) -- Algorithme 1/2 en ligne avec
# optimisme, dictionnaire, extension arriere (Section 5 du papier)
# ============================================================
V_MAX = R_max / (1 - gamma)


class DolarAgent:
    """DOLAR pratique (look-ahead PARFAIT) : meme structure algorithmique que
    le papier (optimisme cout+transition, planificateur a dictionnaire,
    extension arriere en ligne), avec des bonus de confiance de type
    Hoeffding SIMPLIFIES (pas les constantes de Bernstein/Hellinger exactes
    du papier -- trop conservatrices pour une demo numerique, cf. le meme
    choix deja fait pour RPTAS dans le notebook precedent)."""

    def __init__(self, ell, seed=0, M_cap=40, K_cap=150):
        self.ell = ell
        self.rng = np.random.default_rng(seed)
        self.M_cap, self.K_cap = M_cap, K_cap
        self.trans_counts = np.full((N_EXO, N_EXO), 1e-6)
        self.N = 0
        self.reward_sum = np.zeros((N_EXO, N_SoC, N_a))
        self.reward_cnt = np.zeros((N_EXO, N_SoC, N_a))
        self.r_plus = np.zeros((N_EXO, N_SoC, N_a))          # optimiste (bas) pour un cout
        self.D = None

    def _update_reward_plus(self):
        n = np.maximum(self.reward_cnt, 1)
        mean = self.reward_sum / n
        bonus = R_max * np.sqrt(2 * np.log(2 + self.N) / n)
        lcb = np.clip(mean - bonus, 0, R_max)
        lcb = np.where(self.reward_cnt > 0, lcb, 0.0)         # non visite -> optimiste = 0
        self.r_plus = np.maximum(self.r_plus, lcb)            # ne peut que croitre (converge vers la verite)

    def _bonus_transition(self):
        return min(1.0, np.sqrt(2 * np.log(2 + self.N) / max(self.N, 1)))

    def maybe_update(self):
        trigger = (self.D is None) or (self.N > 0 and (self.N & (self.N - 1)) == 0)
        if not trigger:
            return
        self._update_reward_plus()
        Phat = self.trans_counts / self.trans_counts.sum(axis=1, keepdims=True)
        M = int(np.clip(self.N, 8, self.M_cap))
        Zmat = np.zeros((M, N_EXO, N_EXO))
        for j in range(M):
            nxt = np.array([self.rng.choice(N_EXO, p=Phat[e]) for e in range(N_EXO)])
            Zmat[j, np.arange(N_EXO), nxt] = 1.0
        K = int(np.clip(2 / (1 - gamma), 30, self.K_cap))
        v = self._optimistic_vi(Zmat, K)
        self.D = dict(Z=Zmat, v=v, M=M)

    def _optimistic_vi(self, Zmat, K):
        M = Zmat.shape[0]
        v = np.full((S_PHYS, M ** self.ell), V_MAX)
        for _ in range(K):
            v = self._optimistic_bellman(v, Zmat, M)
        return v

    def _optimistic_bellman(self, v, Zmat, M):
        ell = self.ell
        Mpow = M ** (ell - 1)
        vr = v.reshape(S_PHYS, Mpow, M)
        mean, std = vr.mean(axis=2), vr.std(axis=2)
        bonus = self._bonus_transition()
        lcb = np.clip(mean - 2 * bonus * std - V_MAX * bonus ** 2, 0, V_MAX)
        G2 = lcb.reshape(N_EXO, N_SoC * Mpow)
        vn = np.empty_like(v)
        for i0 in range(M):
            GB = gamma * (Zmat[i0] @ G2).reshape(N_EXO, N_SoC, Mpow)
            cand = np.stack([self.r_plus[:, :, a][:, :, None] + GB[:, SOC_NEXT[:, a], :]
                              for a in range(N_a)], axis=2)
            vn[:, i0 * Mpow:(i0 + 1) * Mpow] = cand.min(axis=2).reshape(S_PHYS, -1)
        return np.clip(vn, 0, V_MAX)

    def act(self, e_s, k_s, window):
        ell, M = self.ell, self.D['M']
        kaps = []
        for nxt in window:
            K = np.zeros((N_EXO, N_EXO))
            K[np.arange(N_EXO), nxt] = 1.0
            kaps.append(K)
        U = self.D['v']
        for kk in range(ell - 1, 0, -1):
            Nk = M ** kk
            H = U.reshape(S_PHYS, Nk, M).mean(axis=2).reshape(N_EXO, N_SoC * Nk)
            GB = gamma * (kaps[kk] @ H).reshape(N_EXO, N_SoC, Nk)
            cand = np.stack([self.r_plus[:, :, a][:, :, None] + GB[:, SOC_NEXT[:, a], :]
                              for a in range(N_a)], axis=2)
            U = cand.min(axis=2).reshape(S_PHYS, -1)
        GB = gamma * (kaps[0] @ U.mean(axis=1).reshape(N_EXO, N_SoC))
        scores = self.r_plus[e_s, k_s, :] + GB[e_s, SOC_NEXT[k_s, :]]
        return int(np.argmin(scores))

    def observe(self, e_before, k_before, a, theta_t, cost):
        self.trans_counts[np.arange(N_EXO), theta_t] += 1
        self.N += 1
        self.reward_sum[e_before, k_before, a] += cost
        self.reward_cnt[e_before, k_before, a] += 1


In [ ]:
# ============================================================
# ETAPE 4 : NOISY-DOLAR -- look-ahead LOCALEMENT CORROMPU (Appendice du
# papier). Canal de corruption simple : erasure-to-uniform independante
# par entree, parametree par eta in [0,1] (eta=0 -> look-ahead parfait,
# doit redonner un comportement proche de DOLAR -- verifie plus bas).
# ============================================================

# ============================================================
# Canal de corruption locale : avec proba (1-eta) l'indice revele est le
# VRAI prochain bin ; avec proba eta, c'est un bin UNIFORME au hasard
# (erasure-to-uniform), independamment par entree -- correspond a I(.|s,a,s')
# du papier (canal local, factorise coordonnee par coordonnee).
# ============================================================

class NoisyWindEnv:
    def __init__(self, ell, eta, seed=0, e0=0, k0=0, reward_noise=0.05):
        self.ell, self.eta = ell, eta
        self.rng = np.random.default_rng(seed)
        self.e, self.k = e0, k0
        self.reward_noise = reward_noise
        self.window = [self._draw_true_table() for _ in range(ell)]      # tables LATENTES (theta)
        self.noisy_window = [self._corrupt(t) for t in self.window]       # tables BRUITEES (theta~)

    def _draw_true_table(self):
        return np.array([self.rng.choice(N_EXO, p=P_EXO_TRUE[e]) for e in range(N_EXO)])

    def _corrupt(self, table):
        out = table.copy()
        flip = self.rng.random(N_EXO) < self.eta
        out[flip] = self.rng.integers(0, N_EXO, flip.sum())
        return out

    def observe(self):
        """Renvoie l'etat physique et la fenetre BRUITEE (ce que l'agent voit)."""
        return (self.e, self.k), list(self.noisy_window)

    def step(self, a):
        theta0 = self.window[0]              # LATENTE : gouverne la vraie transition
        z0 = self.noisy_window[0][self.e]    # indice BRUITE au bin courant (ce que l'agent A VU)
        true_cost = R3_TRUE[self.e, self.k, a]
        noisy_cost = float(np.clip(
            true_cost + self.rng.uniform(-self.reward_noise, self.reward_noise), 0, R_max))
        e_before, k_before = self.e, self.k
        e_next = int(theta0[self.e])          # VRAI prochain bin (jamais observe directement ailleurs)
        k_next = int(SOC_NEXT[self.k, a])
        self.window.pop(0); self.noisy_window.pop(0)
        t_new = self._draw_true_table()
        z_new = self._corrupt(t_new)
        self.window.append(t_new)
        self.noisy_window.append(z_new)
        self.e, self.k = e_next, k_next
        # ce que l'agent peut effectivement enregistrer : le contexte local
        # (e_before, z0), le VRAI bin d'arrivee e_next (deductible de S_{t+1}),
        # et la table BRUITEE fraichement revelee z_new (info PLEINE, pour la
        # loi produit -- distincte du posterieur local, qui lui reste bandit).
        return e_before, k_before, z0, e_next, a, noisy_cost, z_new


class NoisyDolarAgent:
    """Noisy-DOLAR pratique. Deux mecanismes d'apprentissage distincts,
    exactement comme dans le papier :
      - loi produit du bruit Qtilde_hat[e,:] : info PLEINE (chaque table
        fraiche revelee donne une observation pour TOUS les bins e a la
        fois) -> apprentissage RAPIDE.
      - posterieur local hat_P(.|e,z) : info BANDIT (seulement mis a jour
        quand l'agent occupe REELLEMENT le bin e et observe REELLEMENT
        l'indice z) -> apprentissage LENT, retombe sur UNIFORME si (e,z)
        jamais visite (comme le papier : "harmless because the optimistic
        estimate will be clipped").
    """

    def __init__(self, ell, seed=0, M_cap=40, K_cap=150):
        self.ell = ell
        self.rng = np.random.default_rng(seed)
        self.M_cap, self.K_cap = M_cap, K_cap
        self.noisy_marg_counts = np.full((N_EXO, N_EXO), 1e-6)      # Qtilde_hat[e, z]
        self.local_counts = np.full((N_EXO, N_EXO, N_EXO), 1e-9)    # hat_P(e'|e,z) numerateur
        self.local_visited = np.zeros((N_EXO, N_EXO), dtype=bool)
        self.N = 0
        self.reward_sum = np.zeros((N_EXO, N_SoC, N_a))
        self.reward_cnt = np.zeros((N_EXO, N_SoC, N_a))
        self.r_plus = np.zeros((N_EXO, N_SoC, N_a))
        self.D = None

    def _update_reward_plus(self):
        n = np.maximum(self.reward_cnt, 1)
        mean = self.reward_sum / n
        bonus = R_max * np.sqrt(2 * np.log(2 + self.N) / n)
        lcb = np.clip(mean - bonus, 0, R_max)
        lcb = np.where(self.reward_cnt > 0, lcb, 0.0)
        self.r_plus = np.maximum(self.r_plus, lcb)

    def _local_posterior_row(self, e, z):
        if self.local_visited[e, z]:
            row = self.local_counts[e, z]
            return row / row.sum()
        return np.full(N_EXO, 1.0 / N_EXO)          # repli uniforme (contexte jamais visite)

    def _bonus_transition(self):
        return min(1.0, np.sqrt(2 * np.log(2 + self.N) / max(self.N, 1)))

    def maybe_update(self):
        trigger = (self.D is None) or (self.N > 0 and (self.N & (self.N - 1)) == 0)
        if not trigger:
            return
        self._update_reward_plus()
        Qhat = self.noisy_marg_counts / self.noisy_marg_counts.sum(axis=1, keepdims=True)
        M = int(np.clip(self.N, 8, self.M_cap))
        Zmat = np.zeros((M, N_EXO, N_EXO))                # noyaux POSTERIEURS (pas one-hot)
        for j in range(M):
            z_row = np.array([self.rng.choice(N_EXO, p=Qhat[e]) for e in range(N_EXO)])
            for e in range(N_EXO):
                Zmat[j, e] = self._local_posterior_row(e, z_row[e])
        K = int(np.clip(2 / (1 - gamma), 30, self.K_cap))
        v = self._optimistic_vi(Zmat, K)
        self.D = dict(Z=Zmat, v=v, M=M)

    def _optimistic_vi(self, Zmat, K):
        M = Zmat.shape[0]
        v = np.full((S_PHYS, M ** self.ell), V_MAX)
        for _ in range(K):
            v = self._optimistic_bellman(v, Zmat, M)
        return v

    def _optimistic_bellman(self, v, Zmat, M):
        ell = self.ell
        Mpow = M ** (ell - 1)
        vr = v.reshape(S_PHYS, Mpow, M)
        mean, std = vr.mean(axis=2), vr.std(axis=2)
        bonus = self._bonus_transition()
        lcb = np.clip(mean - 2 * bonus * std - V_MAX * bonus ** 2, 0, V_MAX)
        G2 = lcb.reshape(N_EXO, N_SoC * Mpow)
        vn = np.empty_like(v)
        for i0 in range(M):
            GB = gamma * (Zmat[i0] @ G2).reshape(N_EXO, N_SoC, Mpow)
            cand = np.stack([self.r_plus[:, :, a][:, :, None] + GB[:, SOC_NEXT[:, a], :]
                              for a in range(N_a)], axis=2)
            vn[:, i0 * Mpow:(i0 + 1) * Mpow] = cand.min(axis=2).reshape(S_PHYS, -1)
        return np.clip(vn, 0, V_MAX)

    def act(self, e_s, k_s, noisy_window):
        ell, M = self.ell, self.D['M']
        kaps = []
        for z_row in noisy_window:
            K = np.stack([self._local_posterior_row(e, int(z_row[e])) for e in range(N_EXO)])
            kaps.append(K)
        U = self.D['v']
        for kk in range(ell - 1, 0, -1):
            Nk = M ** kk
            H = U.reshape(S_PHYS, Nk, M).mean(axis=2).reshape(N_EXO, N_SoC * Nk)
            GB = gamma * (kaps[kk] @ H).reshape(N_EXO, N_SoC, Nk)
            cand = np.stack([self.r_plus[:, :, a][:, :, None] + GB[:, SOC_NEXT[:, a], :]
                              for a in range(N_a)], axis=2)
            U = cand.min(axis=2).reshape(S_PHYS, -1)
        GB = gamma * (kaps[0] @ U.mean(axis=1).reshape(N_EXO, N_SoC))
        scores = self.r_plus[e_s, k_s, :] + GB[e_s, SOC_NEXT[k_s, :]]
        return int(np.argmin(scores))

    def observe(self, e_before, k_before, z0, e_next, a, cost, fresh_noisy_table=None):
        self.local_counts[e_before, z0, e_next] += 1
        self.local_visited[e_before, z0] = True
        if fresh_noisy_table is not None:
            self.noisy_marg_counts[np.arange(N_EXO), fresh_noisy_table] += 1
        self.N += 1
        self.reward_sum[e_before, k_before, a] += cost
        self.reward_cnt[e_before, k_before, a] += 1


In [ ]:
# ============================================================
# ETAPE 5 : execution -- courbes d'apprentissage (regret cumule)
# ============================================================
# Regret = somme_t [ Q*(X_t,A_t) - V*(X_t) ] (ecart de Bellman, eq.
# operationnalisant Reg_ell(T) du papier), calcule ici EXACTEMENT car on
# connait P_EXO_TRUE/R3_TRUE (verite terrain utilisee UNIQUEMENT pour
# l'evaluation -- jamais donnee a l'agent).


def run_dolar(ell, T, seed):
    env = WindEnv(ell, seed=seed)
    agent = DolarAgent(ell, seed=seed)
    agent.maybe_update()
    reg = np.empty(T)
    for t in range(T):
        (e, k), w = env.observe()
        a = agent.act(e, k, w)
        e_b, k_b, theta_t, cost = env.step(a)
        agent.observe(e_b, k_b, a, theta_t, cost)
        agent.maybe_update()
        reg[t] = Q_true[e_b, k_b, a] - V_true[e_b, k_b]
    return reg


def run_noisy(ell, eta, T, seed):
    env = NoisyWindEnv(ell, eta, seed=seed)
    agent = NoisyDolarAgent(ell, seed=seed)
    agent.maybe_update()
    reg = np.empty(T)
    for t in range(T):
        (e, k), w = env.observe()
        a = agent.act(e, k, w)
        e_b, k_b, z0, e_n, a, cost, z_new = env.step(a)
        agent.observe(e_b, k_b, z0, e_n, a, cost, fresh_noisy_table=z_new)
        agent.maybe_update()
        reg[t] = Q_true[e_b, k_b, a] - V_true[e_b, k_b]
    return reg


T = 5000
SEEDS = [1, 2, 3, 4, 5]
ELL = 2
ETA = 0.3          # niveau de corruption pour Noisy-DOLAR

print("Simulation DOLAR (look-ahead parfait) : %d graines x %d pas..." % (len(SEEDS), T))
R_dolar = np.stack([run_dolar(ELL, T, s) for s in SEEDS])

print("Simulation Noisy-DOLAR (eta=%.2f) : %d graines x %d pas..." % (ETA, len(SEEDS), T))
R_noisy = np.stack([run_noisy(ELL, ETA, T, s) for s in SEEDS])

# --- Sanity check : eta=0 doit redonner un comportement proche de DOLAR ---
print("Verification eta=0 (doit ressembler a DOLAR) : %d graines x %d pas..." % (len(SEEDS), 1500))
R_check = np.stack([run_noisy(ELL, 0.0, 1500, s) for s in SEEDS])
print("  regret moyen/pas sur les 300 derniers pas -- eta=0: %.4f" % R_check[:, -300:].mean())

print("\nRegret cumule final (moyenne +/- ecart-type sur les graines) :")
print("  DOLAR (parfait)      : %.1f +/- %.1f" % (
    np.cumsum(R_dolar, axis=1)[:, -1].mean(), np.cumsum(R_dolar, axis=1)[:, -1].std()))
print("  Noisy-DOLAR (eta=%.2f): %.1f +/- %.1f" % (
    ETA, np.cumsum(R_noisy, axis=1)[:, -1].mean(), np.cumsum(R_noisy, axis=1)[:, -1].std()))


In [ ]:
# ============================================================
# ETAPE 6 : figures -- courbes d'apprentissage
# ============================================================
cum_dolar = np.cumsum(R_dolar, axis=1)
cum_noisy = np.cumsum(R_noisy, axis=1)
t_axis = np.arange(1, T + 1)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

# (a) regret CUMULE -- la courbe habituelle pour visualiser un theoreme de
# regret (une croissance en O(sqrt(T)) se voit comme une courbe concave qui
# s'aplatit progressivement, jamais une droite).
ax = axes[0]
mu, sd = cum_dolar.mean(axis=0), cum_dolar.std(axis=0)
ax.plot(t_axis, mu, color='tab:red', label='DOLAR (look-ahead parfait)')
ax.fill_between(t_axis, mu - sd, mu + sd, color='tab:red', alpha=.15)
mu, sd = cum_noisy.mean(axis=0), cum_noisy.std(axis=0)
ax.plot(t_axis, mu, color='tab:blue', label='Noisy-DOLAR ($\\eta$=%.2f)' % ETA)
ax.fill_between(t_axis, mu - sd, mu + sd, color='tab:blue', alpha=.15)
# repere visuel : une courbe en c*sqrt(T) ajustee au dernier point de DOLAR
c = cum_dolar.mean(axis=0)[-1] / np.sqrt(T)
ax.plot(t_axis, c * np.sqrt(t_axis), '--', color='gray', lw=1, label='repère $\\propto\\sqrt{T}$')
ax.set_xlabel('pas de temps $T$')
ax.set_ylabel('regret cumule (écart de Bellman)')
ax.set_title('(a) Regret cumulé')
ax.grid(True, ls='--', alpha=.5)
ax.legend(fontsize=8)

# (b) regret MOYEN par pas, fenetre glissante -- montre la VITESSE
# d'apprentissage (doit decroitre vers 0, lentement, en 1/sqrt(t))
ax = axes[1]
window = 300
def rolling_mean(x, w):
    c = np.cumsum(np.insert(x, 0, 0))
    return (c[w:] - c[:-w]) / w

for R, color, label in [(R_dolar, 'tab:red', 'DOLAR (parfait)'),
                         (R_noisy, 'tab:blue', 'Noisy-DOLAR ($\\eta$=%.2f)' % ETA)]:
    rm = np.stack([rolling_mean(r, window) for r in R])
    mu, sd = rm.mean(axis=0), rm.std(axis=0)
    xs = np.arange(window, T + 1)
    ax.plot(xs, mu, color=color, label=label)
    ax.fill_between(xs, mu - sd, mu + sd, color=color, alpha=.15)
ax.set_xlabel('pas de temps $T$')
ax.set_ylabel('regret moyen (fenetre glissante de %d pas)' % window)
ax.set_title("(b) Vitesse d'apprentissage")
ax.grid(True, ls='--', alpha=.5)
ax.legend(fontsize=8)

plt.tight_layout()
plt.show()

print("\nNOTE DE LECTURE : a cette echelle (N_EXO=%d, N_a=%d) et avec des bonus de\n"
      "confiance simplifies (cf. cellule 0), l'ecart QUANTITATIF entre DOLAR et\n"
      "Noisy-DOLAR est faible/bruite -- l'apprentissage de la RECOMPENSE (bandit\n"
      "dans les DEUX cas) domine le regret total et masque la difference de\n"
      "vitesse d'apprentissage de la TRANSITION (rapide, info pleine, pour DOLAR ;\n"
      "lente, info bandit par contexte local, pour Noisy-DOLAR). Ce que montrent\n"
      "fiablement ces courbes : (i) les deux algorithmes APPRENNENT (regret\n"
      "cumule sous-lineaire, coherent avec sqrt(T)) ; (ii) a eta=0, Noisy-DOLAR\n"
      "se comporte comme DOLAR (verification ci-dessus), validant l'implementation." % (N_EXO, N_a))


In [ ]:
# ============================================================
# ETAPE 7 : brancher les VRAIES donnees CAISO (notebook RPTAS_on_BOLA)
# ============================================================
# Ce notebook ne peut pas etre execute directement sur les vraies donnees
# ici (les CSV CAISO et le notebook des auteurs "RL with Prediction_
# simulation.ipynb" ne sont pas presents dans cet environnement). Pour
# reutiliser exactement le meme jeu de donnees que RPTAS_on_BOLA_protocol,
# remplacez la cellule "ETAPE 1" ci-dessus par ce qui suit :
#
#   1. Executez les cellules STEP 0 a STEP 2 du notebook RPTAS_on_BOLA
#      (chargement CSV, discretisation, P_EXO, R3, SOC_NEXT, ACT_EFF,
#      N_EXO=100, N_SoC=21, N_a=9, gamma=0.95) -- soit en executant ce
#      notebook a la suite du notebook RPTAS_on_BOLA dans le MEME kernel,
#      soit en copiant ces cellules ici.
#   2. Renommez P_EXO -> P_EXO_TRUE et R3 -> R3_TRUE (ce sont les memes
#      objets : le noyau exogene et la table de cout ESTIMES sur la
#      trajectoire CAISO d'entrainement -- ici ils jouent le role de la
#      "verite terrain" que DOLAR/Noisy-DOLAR doivent decouvrir en ligne,
#      SANS y avoir acces directement).
#   3. Calculez V_true, Q_true via exact_vi(P_EXO_TRUE, R3_TRUE) (deja
#      fourni ci-dessus) -- c'est l'oracle utilise pour mesurer le regret.
#   4. R_max doit alors valoir max(R3_TRUE) (pas 1.0) ; adaptez les bonus
#      qui en dependent (deja parametres via la variable R_max partout,
#      aucun autre changement necessaire dans DolarAgent/NoisyDolarAgent).
#   5. Avec N_EXO=100 (au lieu de 6), le budget de dictionnaire M_cap et
#      le nombre d'iterations K_cap devront rester MODESTES (le cout par
#      iteration croit en O(N_EXO*(N_a+1)*M^ell), comme deja discute pour
#      RPTAS) -- gardez M_cap<=30-40 pour ell<=2, et reduisez ell si le
#      temps de calcul devient trop important.
#   6. Pour Noisy-DOLAR, le canal de corruption eta=0.3 est un choix
#      arbitraire ici ; pour coller au protocole du notebook RPTAS_on_BOLA
#      (bruit multiplicatif gaussien sur le prix/mismatch), on pourrait
#      remplacer le canal "erasure-to-uniform" par une corruption
#      construite a partir de kernel_from_likelihood/likelihood_and_rbar
#      -- une extension naturelle mais non implementee ici par souci de
#      temps ; le canal simple utilise suffit a illustrer la distinction
#      structurelle info-pleine (DOLAR) vs info-bandit-locale (Noisy-DOLAR)
#      qui est le point theorique central de l'appendice bruite du papier.

print("Voir les instructions ci-dessus pour reutiliser les vraies donnees CAISO.")
